# NB-Ramen — 252 runs với dữ liệu Hugging Face

Bật Internet, chọn GPU T4, rồi Save Version → Save & Run All để chạy nền.
Giữ 252 runs, batch=100, block=64, 400 source samples/domain và một GPU.
`MAX_RUNS=0`, `SESSION_HOURS=0` cho phép chạy mọi run còn thiếu đến giới hạn Kaggle.

Notebook tải ảnh từ Hugging Face, chuyển về NumPy và đối chiếu cả 20 file
với checksum của CIFAR-100-C gốc. Nguồn Hugging Face được ghi rõ trong evidence;
không khai rằng đã tải hay đo checksum archive Zenodo.

Source nền: `26a7cd7c847b6630841dbae58067e5bb124f2f9d`. Patch acquisition được nhúng trong notebook và tạo commit
local xác định `dc3cbf02215cff59046f048e6580dd8f8d40af89`. Mỗi phiên đều chạy tests trên source đã patch.
Các thuật toán, cấu hình, CLIP và ma trận không đổi. Dùng checkpoint riêng của
notebook này để resume. Chưa có kết quả CUDA đầy đủ cho bản acquisition này.


In [ ]:
import os, sys, json, subprocess, shutil, uuid, importlib.util
from datetime import datetime, timezone
from pathlib import Path

REVISION = "26a7cd7c847b6630841dbae58067e5bb124f2f9d"
WORK = Path("/kaggle/working")
REPO = WORK / "NB-Ramen-HF"
PYTHON = Path("/tmp/nb-ramen-venv/bin/python")
DATA = Path("/tmp/nb-ramen-hf-data")
EVIDENCE = WORK / "nb-ramen-full-hf-dc3cbf0"
# Chỉ sửa ba tùy chọn này và ARCHIVE_INPUT nếu cần.
RESUME_ARCHIVE = ""  # Ví dụ /kaggle/input/my-full-checkpoint/nb-ramen-full-hf-dc3cbf0.zip
MAX_RUNS = 0        # 0 = mọi run còn thiếu; ví dụ 7 = tối đa 7 run mới trong phiên
SESSION_HOURS = 0.0 # 0 = không dừng theo thời gian; kiểm tra ngân sách giữa các run
SESSION_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "-" + uuid.uuid4().hex[:8]
RUNTIME = EVIDENCE / "runtime" / SESSION_ID
# ARCHIVE_INPUT không dùng trong notebook Hugging Face này.
ARCHIVE_INPUT = ""
WORK.mkdir(parents=True, exist_ok=True)
checkpoint_script = WORK / "nb-ramen-full-checkpoint.py"
checkpoint_script.write_text('"""Atomic ZIP checkpoints and non-overwriting restore for Kaggle campaigns."""\n\nimport argparse\nimport hashlib\nimport json\nfrom pathlib import Path, PurePosixPath\nimport shutil\nimport stat\nimport tempfile\nimport zipfile\n\n\ndef checkpoint(evidence):\n    evidence = Path(evidence).resolve()\n    if not evidence.is_dir():\n        raise FileNotFoundError(evidence)\n    archive = evidence.with_suffix(".zip")\n    partial = archive.with_suffix(".zip.part")\n    # Keep the previous complete ZIP until the replacement is closed successfully.\n    try:\n        with zipfile.ZipFile(partial, "w", zipfile.ZIP_DEFLATED, compresslevel=1) as output:\n            for path in sorted(evidence.rglob("*")):\n                if path.is_symlink():\n                    raise ValueError(f"Evidence must not contain symlinks: {path}")\n                if path.is_file():\n                    output.write(path, path.relative_to(evidence.parent).as_posix())\n        partial.replace(archive)\n    except BaseException:\n        partial.unlink(missing_ok=True)\n        raise\n    return archive\n\n\ndef restore(archive, evidence):\n    archive, evidence = Path(archive), Path(evidence)\n    with archive.open("rb") as handle:\n        digest = hashlib.file_digest(handle, "sha256").hexdigest()\n    marker_name = "restored-archive.json"\n    if evidence.exists():\n        marker = evidence / marker_name\n        if marker.is_file() and json.loads(marker.read_text()).get("sha256") == digest:\n            return  # Setup rerun: retain all progress since the same restore.\n        raise FileExistsError(\n            "Evidence already exists. Clear RESUME_ARCHIVE to use it; restore never merges or overwrites runs."\n        )\n    evidence.parent.mkdir(parents=True, exist_ok=True)\n    with zipfile.ZipFile(archive) as source:\n        seen = set()\n        for entry in source.infolist():\n            parts = PurePosixPath(entry.filename).parts\n            mode = entry.external_attr >> 16\n            normalized = PurePosixPath(entry.filename).as_posix()\n            if (not parts or parts[0] != evidence.name or ".." in parts\n                    or "\\\\" in entry.filename or PurePosixPath(entry.filename).is_absolute()\n                    or stat.S_ISLNK(mode) or normalized in seen):\n                raise ValueError(f"Invalid checkpoint member: {entry.filename}")\n            seen.add(normalized)\n        if not seen:\n            raise ValueError("Checkpoint is empty")\n        required = sum(entry.file_size for entry in source.infolist())\n        if required >= shutil.disk_usage(evidence.parent).free:\n            raise OSError("Insufficient disk space to restore this checkpoint")\n        with tempfile.TemporaryDirectory(prefix="restore-full-", dir=evidence.parent) as tmp:\n            # All paths/types were checked before any extraction. CRC errors abort staging.\n            source.extractall(tmp)\n            staged = Path(tmp) / evidence.name\n            if not staged.is_dir():\n                raise ValueError("Checkpoint root must be a directory")\n            (staged / marker_name).write_text(json.dumps({"sha256": digest}, indent=2) + "\\n")\n            staged.rename(evidence)\n\n\nif __name__ == "__main__":\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("action", choices=("save", "restore"))\n    parser.add_argument("evidence", type=Path)\n    parser.add_argument("--archive", type=Path)\n    args = parser.parse_args()\n    if args.action == "restore":\n        if args.archive is None:\n            parser.error("restore requires --archive")\n        restore(args.archive, args.evidence)\n        print("Checkpoint restored; each run still requires strict validation.")\n    else:\n        print(checkpoint(args.evidence))\n')
if RESUME_ARCHIVE.strip():
    subprocess.run([sys.executable, str(checkpoint_script), "restore", str(EVIDENCE),
                    "--archive", RESUME_ARCHIVE], check=True)
RUNTIME.mkdir(parents=True, exist_ok=True)
env = dict(os.environ, HF_HUB_DISABLE_IMPLICIT_TOKEN="1", HF_HUB_DOWNLOAD_TIMEOUT="60", PYTHONPATH=str(REPO / "src"),
           RAMEN_REPOSITORY=str(REPO), RAMEN_REVISION=REVISION,
           RAMEN_DATA_ROOT=str(DATA), RAMEN_EVIDENCE_ROOT=str(EVIDENCE),
           RAMEN_RUNTIME_ROOT=str(RUNTIME),
           RAMEN_ARCHIVE_INPUT=ARCHIVE_INPUT, CUDA_VISIBLE_DEVICES="0",
           PYTHONUNBUFFERED="1", UV_CACHE_DIR="/tmp/nb-ramen-uv-cache")

process_script = RUNTIME / "full-run-process.py"
process_script.write_text('"""Stream subprocess logs and stop the whole experiment group on interruption."""\n\nfrom contextlib import nullcontext\nimport os\nfrom pathlib import Path\nimport signal\nimport subprocess\n\n\ndef stop_group(process):\n    try:\n        os.killpg(process.pid, signal.SIGTERM)\n    except ProcessLookupError:\n        pass\n    try:\n        process.wait(timeout=15)\n    except subprocess.TimeoutExpired:\n        pass\n    # The parent can exit before a descendant; clear surviving group members too.\n    try:\n        os.killpg(process.pid, signal.SIGKILL)\n    except ProcessLookupError:\n        pass\n    process.wait()\n\n\ndef run_logged(argv, *, env=None, log=None, cwd=None):\n    argv = [str(value) for value in argv]\n    with (Path(log).open("w") if log is not None else nullcontext(None)) as handle:\n        process = subprocess.Popen(argv, cwd=cwd, env=env, stdout=subprocess.PIPE,\n                                   stderr=subprocess.STDOUT, text=True, bufsize=1,\n                                   start_new_session=True)\n        try:\n            for line in process.stdout:\n                if handle is not None:\n                    handle.write(line)\n                    handle.flush()\n                print(line, end="", flush=True)\n            code = process.wait()\n        except BaseException:\n            stop_group(process)\n            raise\n        finally:\n            process.stdout.close()\n    if code:\n        # Also stop descendants of a failed driver before its caller saves a ZIP.\n        stop_group(process)\n        raise subprocess.CalledProcessError(code, argv)\n    return subprocess.CompletedProcess(argv, code)\n')
spec = importlib.util.spec_from_file_location("full_run_process", process_script)
process_module = importlib.util.module_from_spec(spec)
spec.loader.exec_module(process_module)

def run(argv, *, log=None, cwd=None):
    return process_module.run_logged(argv, env=env, log=log, cwd=cwd)

run(["nvidia-smi"], log=RUNTIME / "nvidia-smi.txt")
if not REPO.exists():
    run(["git", "clone", "--branch", "open-world-gradient-memory", "--single-branch",
         "https://github.com/nguyetbinh/NB-Ramen.git", REPO])
actual = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=REPO, text=True).strip()
dirty = subprocess.check_output(["git", "status", "--porcelain"], cwd=REPO, text=True)
if dirty:
    raise RuntimeError("Checkout có thay đổi chưa commit; giữ lại các thay đổi trước khi cập nhật.")
if actual != REVISION:
    run(["git", "fetch", "origin", "open-world-gradient-memory"], cwd=REPO)
    run(["git", "checkout", "--detach", REVISION], cwd=REPO)
    actual = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=REPO, text=True).strip()
assert actual == REVISION
source_patch = RUNTIME / "huggingface-source.patch"
source_patch.write_text('--- a/src/runtime/artifact_provenance.py\n+++ b/src/runtime/artifact_provenance.py\n@@ -16,6 +16,12 @@\n from pathlib import Path, PurePosixPath\n from typing import Any, Iterable, Mapping\n from urllib.parse import urlparse\n+\n+\n+try:\n+    from .cifar100c_huggingface import CIFAR100C_HF_ACQUISITION, verify_huggingface_cifar100c_files\n+except ImportError:\n+    from cifar100c_huggingface import CIFAR100C_HF_ACQUISITION, verify_huggingface_cifar100c_files\n \n \n SCHEMA_VERSION = 1\n@@ -287,8 +293,8 @@\n \n \n def _validate_cifar_acquisition(acquisition: Mapping[str, Any]) -> None:\n-    if dict(acquisition) != CIFAR100C_OFFICIAL_ACQUISITION:\n-        raise ProvenanceError("CIFAR-100-C acquisition does not match the pinned official Zenodo artifact")\n+    if dict(acquisition) not in (CIFAR100C_OFFICIAL_ACQUISITION, CIFAR100C_HF_ACQUISITION):\n+        raise ProvenanceError("CIFAR-100-C acquisition does not match the pinned official Zenodo artifact or pinned Hugging Face mirror")\n \n \n def generate_dataset_provenance(dataset: str, dataset_root: str | Path, *, manifest_path: str | Path | None = None,\n@@ -310,8 +316,11 @@\n     acquisition_record = dict(acquisition or {})\n     if dataset == "cifar100c":\n         _validate_cifar_acquisition(acquisition_record)\n-        acquisition_record["expected_checksum"] = acquisition_record["expected_checksum"].lower()\n-        acquisition_record["actual_checksum"] = acquisition_record["actual_checksum"].lower()\n+        if acquisition_record == CIFAR100C_HF_ACQUISITION:\n+            verify_huggingface_cifar100c_files(root)\n+        else:\n+            acquisition_record["expected_checksum"] = acquisition_record["expected_checksum"].lower()\n+            acquisition_record["actual_checksum"] = acquisition_record["actual_checksum"].lower()\n     payload = {"schema_version": SCHEMA_VERSION, "dataset": dataset, "root": ".", "sidecar": f"{SIDECAR_DIRECTORY}/{sidecar.name}",\n                "acquisition": acquisition_record, "content": {"algorithm": "sha256", "files": records, "root_digest": _content_digest(records)}}\n     _atomic_json(sidecar, payload)\n@@ -386,6 +395,8 @@\n             raise ProvenanceError(f"dataset file size changed: {relative}")\n         if exact and sha256_regular_file(path)["sha256"] != record["sha256"]:\n             raise ProvenanceError(f"dataset file SHA-256 mismatch: {relative}")\n+    if exact and dataset == "cifar100c" and payload.get("acquisition") == CIFAR100C_HF_ACQUISITION:\n+        verify_huggingface_cifar100c_files(root)\n     return {\n         "schema_version": SCHEMA_VERSION,\n         "dataset": dataset,\n--- a/src/runtime/experiment_matrix.py\n+++ b/src/runtime/experiment_matrix.py\n@@ -24,7 +24,7 @@\n try:  # Supports both ``python -m runtime...`` and direct-file invocation.\n     from .preflight import validate_dataset_layout\n     from .artifact_provenance import (\n-        CIFAR100C_OFFICIAL_ACQUISITION,\n+        CIFAR100C_OFFICIAL_ACQUISITION, CIFAR100C_HF_ACQUISITION,\n         SCHEMA_VERSION as ARTIFACT_SCHEMA_VERSION,\n         default_sidecar_path,\n         resolve_clip_model,\n@@ -32,7 +32,7 @@\n except ImportError:  # pragma: no cover - exercised only by direct invocation\n     from preflight import validate_dataset_layout\n     from artifact_provenance import (\n-        CIFAR100C_OFFICIAL_ACQUISITION,\n+        CIFAR100C_OFFICIAL_ACQUISITION, CIFAR100C_HF_ACQUISITION,\n         SCHEMA_VERSION as ARTIFACT_SCHEMA_VERSION,\n         default_sidecar_path,\n         resolve_clip_model,\n@@ -925,6 +925,9 @@\n     expected_sidecar = str(default_sidecar_path(root, dataset_key).absolute())\n     _require_equal(sidecar, expected_sidecar, "manifest.artifacts.dataset.sidecar", run)\n     expected_acquisition = CIFAR100C_OFFICIAL_ACQUISITION if run.dataset == "CIFAR100C" else {}\n+    if run.dataset == "CIFAR100C" and dataset.get("acquisition") == CIFAR100C_HF_ACQUISITION:\n+        expected_acquisition = CIFAR100C_HF_ACQUISITION\n+        _require_equal(dataset.get("file_count"), 20, "manifest.artifacts.dataset.file_count", run)\n     _require_equal(dataset.get("acquisition"), expected_acquisition, "manifest.artifacts.dataset.acquisition", run)\n \n \n--- /dev/null\n+++ b/src/runtime/cifar100c_huggingface.py\n@@ -0,0 +1,62 @@\n+"""Pinned Hugging Face transport with independent original NumPy checksums.\n+\n+The MD5 table is published by TorchUncertainty\'s CIFAR100C implementation:\n+https://torch-uncertainty.github.io/_modules/torch_uncertainty/datasets/classification/cifar/cifar_c.html\n+It identifies the original .npy bytes, including image order and NPY headers.\n+This acquisition does not claim to have downloaded or hashed the Zenodo tar.\n+"""\n+\n+from pathlib import Path\n+\n+HF_REPOSITORY = "WNJXYK/TTA-CIFAR-100-C"\n+HF_REVISION = "a12f0bcc1da33fa26d8c76ce8c1fb32e6f913bea"\n+CIFAR100C_NPY_MD5 = {\n+    "brightness.npy": "f22d7195aecd6abb541e27fca230c171",\n+    "contrast.npy": "322bb385f1d05154ee197ca16535f71e",\n+    "defocus_blur.npy": "d923e3d9c585a27f0956e2f2ad832564",\n+    "elastic_transform.npy": "a0792bd6581f6810878be71acedfc65a",\n+    "fog.npy": "4efc7ebd5e82b028bdbe13048e3ea564",\n+    "frost.npy": "3a39c6823bdfaa0bf8b12fe7004b8117",\n+    "gaussian_blur.npy": "5204ba0d557839772ef5a4196a052c3e",\n+    "gaussian_noise.npy": "ecc4d366eac432bdf25c024086f5e97d",\n+    "glass_blur.npy": "0bf384f38e5ccbf8dd479d9059b913e1",\n+    "impulse_noise.npy": "3b3c210ddfa0b5cb918ff4537a429fef",\n+    "jpeg_compression.npy": "c851b7f1324e1d2ffddeb76920576d11",\n+    "labels.npy": "bb4026e9ce52996b95f439544568cdb2",\n+    "motion_blur.npy": "732a7e2e54152ff97c742d4c388c5516",\n+    "pixelate.npy": "96c00c60f144539e14cffb02ddbd0640",\n+    "saturate.npy": "c0697e9fdd646916a61e9c312c77bf6b",\n+    "shot_noise.npy": "b0a1fa6e1e465a747c1b204b1914048a",\n+    "snow.npy": "0237be164583af146b7b144e73b43465",\n+    "spatter.npy": "12ccf41d62564d36e1f6a6ada5022728",\n+    "speckle_noise.npy": "e3f215b1a0f9fd9fd6f0d1cf94a7ce99",\n+    "zoom_blur.npy": "0204613400c034a81c4830d5df81cb82",\n+}\n+CIFAR100C_HF_ACQUISITION = {\n+    "publisher": "Hugging Face community mirror",\n+    "repository": HF_REPOSITORY,\n+    "revision": HF_REVISION,\n+    "url": f"https://huggingface.co/datasets/{HF_REPOSITORY}/tree/{HF_REVISION}",\n+    "upstream_doi": "10.5281/zenodo.3555552",\n+    "verification": "reconstructed_original_npy_md5",\n+    "files": dict(CIFAR100C_NPY_MD5),\n+}\n+\n+\n+def verify_huggingface_cifar100c_files(root):\n+    """Check all twenty original files, not just mirror sizes or self-reported hashes."""\n+    try:\n+        from .artifact_provenance import ProvenanceError, _walk_regular_files, checksum_regular_file\n+    except ImportError:\n+        from artifact_provenance import ProvenanceError, _walk_regular_files, checksum_regular_file\n+\n+    files = dict(_walk_regular_files(Path(root)))\n+    if set(files) != set(CIFAR100C_NPY_MD5):\n+        raise ProvenanceError("Hugging Face CIFAR-100-C requires exactly the twenty original NPY files")\n+    actual = {}\n+    for name, expected in CIFAR100C_NPY_MD5.items():\n+        digest = checksum_regular_file(files[name], "md5")["checksum"]\n+        if digest != expected:\n+            raise ProvenanceError(f"Original CIFAR-100-C NPY checksum mismatch: {name}")\n+        actual[name] = digest\n+    return {**CIFAR100C_HF_ACQUISITION, "files": actual}\n--- /dev/null\n+++ b/tests/test_cifar100c_huggingface.py\n@@ -0,0 +1,84 @@\n+import copy\n+import json\n+from pathlib import Path\n+import tempfile\n+import unittest\n+\n+from src.runtime import artifact_provenance as provenance\n+from src.runtime.cifar100c_huggingface import (\n+    CIFAR100C_HF_ACQUISITION, CIFAR100C_NPY_MD5, verify_huggingface_cifar100c_files,\n+)\n+\n+\n+class HuggingFaceAcquisitionTests(unittest.TestCase):\n+    def test_pinned_acquisitions_are_distinct_and_accepted(self):\n+        self.assertNotEqual(CIFAR100C_HF_ACQUISITION, provenance.CIFAR100C_OFFICIAL_ACQUISITION)\n+        provenance._validate_cifar_acquisition(CIFAR100C_HF_ACQUISITION)\n+        provenance._validate_cifar_acquisition(provenance.CIFAR100C_OFFICIAL_ACQUISITION)\n+        self.assertNotIn("actual_checksum", CIFAR100C_HF_ACQUISITION)\n+\n+    def test_changed_revision_url_or_checksum_is_rejected(self):\n+        for field in ("revision", "url", "files"):\n+            with self.subTest(field=field):\n+                bad = copy.deepcopy(CIFAR100C_HF_ACQUISITION)\n+                bad[field] = {} if field == "files" else "changed"\n+                with self.assertRaises(provenance.ProvenanceError):\n+                    provenance._validate_cifar_acquisition(bad)\n+\n+    def test_generation_checks_original_bytes_before_writing_sidecar(self):\n+        with tempfile.TemporaryDirectory() as tmp:\n+            root = Path(tmp)\n+            for name in CIFAR100C_NPY_MD5:\n+                (root / name).write_bytes(b"not the official data")\n+            with self.assertRaisesRegex(provenance.ProvenanceError, "checksum mismatch"):\n+                provenance.generate_cifar100c_provenance(root, acquisition=CIFAR100C_HF_ACQUISITION)\n+            self.assertFalse(provenance.default_sidecar_path(root, "cifar100c").exists())\n+\n+    def test_missing_or_extra_file_is_rejected(self):\n+        with tempfile.TemporaryDirectory() as tmp:\n+            root = Path(tmp)\n+            with self.assertRaisesRegex(provenance.ProvenanceError, "twenty"):\n+                verify_huggingface_cifar100c_files(root)\n+            for name in CIFAR100C_NPY_MD5:\n+                (root / name).write_bytes(b"")\n+            (root / "extra.npy").write_bytes(b"")\n+            with self.assertRaisesRegex(provenance.ProvenanceError, "twenty"):\n+                verify_huggingface_cifar100c_files(root)\n+\n+    def test_symlink_is_rejected(self):\n+        with tempfile.TemporaryDirectory() as tmp:\n+            root = Path(tmp) / "data"\n+            root.mkdir()\n+            target = Path(tmp) / "target"\n+            target.write_bytes(b"data")\n+            (root / "labels.npy").symlink_to(target)\n+            with self.assertRaises(provenance.ProvenanceError):\n+                verify_huggingface_cifar100c_files(root)\n+\n+    def test_strict_matrix_validation_accepts_pinned_hf_and_rejects_altered_receipt(self):\n+        from src.runtime.experiment_matrix import build_experiment_matrix, validate_completed_run, IncompleteRunError\n+        from tests.test_experiment_matrix import _write_valid_evidence\n+\n+        with tempfile.TemporaryDirectory() as tmp:\n+            run = build_experiment_matrix(datasets=("CIFAR100C",), methods=("NoAdapt",),\n+                                          streams=("block",), seeds=(0,), device="cpu",\n+                                          evidence_dir=Path(tmp))[0]\n+            _write_valid_evidence(run)\n+            path = run.run_dir / "manifest.json"\n+            manifest = json.loads(path.read_text())\n+            manifest["artifacts"]["dataset"].update(\n+                acquisition=copy.deepcopy(CIFAR100C_HF_ACQUISITION), file_count=20,\n+            )\n+            path.write_text(json.dumps(manifest))\n+            validate_completed_run(run)\n+            for field, value in (("file_count", 19), ("acquisition", {})):\n+                with self.subTest(field=field):\n+                    bad = copy.deepcopy(manifest)\n+                    bad["artifacts"]["dataset"][field] = value\n+                    path.write_text(json.dumps(bad))\n+                    with self.assertRaises(IncompleteRunError):\n+                        validate_completed_run(run)\n+\n+\n+if __name__ == "__main__":\n+    unittest.main()\n')
bootstrap_script = RUNTIME / "bootstrap-huggingface-source.py"
bootstrap_script.write_text('"""Apply the bundled acquisition patch as a reproducible, local-only Git commit."""\n\nimport os\nfrom pathlib import Path\nimport subprocess\nimport sys\n\nBASE_REVISION = "26a7cd7c847b6630841dbae58067e5bb124f2f9d"\n\n\ndef bootstrap(repo, patch):\n    repo, patch = Path(repo), Path(patch).resolve()\n    head = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=repo, text=True).strip()\n    dirty = subprocess.check_output(["git", "status", "--porcelain"], cwd=repo, text=True)\n    if head != BASE_REVISION or dirty:\n        raise RuntimeError("Apply Hugging Face support only to the clean pinned base revision")\n    subprocess.run(["git", "apply", "--check", str(patch)], cwd=repo, check=True)\n    subprocess.run(["git", "apply", "--index", str(patch)], cwd=repo, check=True)\n    commit_env = dict(os.environ,\n        GIT_AUTHOR_NAME="NB-Ramen", GIT_AUTHOR_EMAIL="nb-ramen@localhost",\n        GIT_COMMITTER_NAME="NB-Ramen", GIT_COMMITTER_EMAIL="nb-ramen@localhost",\n        GIT_AUTHOR_DATE="2026-09-14T00:00:00+00:00", GIT_COMMITTER_DATE="2026-09-14T00:00:00+00:00",\n    )\n    subprocess.run(["git", "-c", "commit.gpgsign=false", "commit", "-m",\n                    "feat(runtime): verify Hugging Face CIFAR-100-C acquisition"],\n                   cwd=repo, env=commit_env, check=True)\n    print(subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=repo, text=True).strip())\n\n\nif __name__ == "__main__":\n    bootstrap(*sys.argv[1:])\n')
run([sys.executable, bootstrap_script, REPO, source_patch], cwd=REPO, log=RUNTIME / "source-patch.log")
actual = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=REPO, text=True).strip()
assert actual == 'dc3cbf02215cff59046f048e6580dd8f8d40af89', "Hugging Face source commit is not reproducible"
REVISION = actual
env["RAMEN_REVISION"] = REVISION
(RUNTIME / "git-head.txt").write_text(actual + "\n")
(RUNTIME / "git-status.txt").write_text(dirty)

run([sys.executable, "-m", "pip", "install", "--quiet", "uv"])
if not PYTHON.exists():
    run([sys.executable, "-m", "uv", "venv", "--python", "3.11", PYTHON.parent.parent])
run([sys.executable, "-m", "uv", "pip", "install", "--python", PYTHON, "pip==24.2"])
run([PYTHON, "-m", "pip", "install", "--no-cache-dir", "torch==2.4.1", "torchvision==0.19.1",
     "--index-url", "https://download.pytorch.org/whl/cu121"])
run([PYTHON, "-m", "pip", "install", "--no-cache-dir", "numpy==1.26.4", "pillow==10.4.0",
     "pyyaml==6.0.2", "tqdm==4.66.5", "pyarrow==18.1.0", "huggingface-hub==0.26.2",
     "git+https://github.com/openai/CLIP.git@d05afc436d78f1c48dc0dbf8e5980a9d471f35f6"])
run([PYTHON, "-m", "pip", "check"], log=RUNTIME / "pip-check.txt")
run([PYTHON, "-m", "pip", "freeze"], log=RUNTIME / "pip-freeze.txt")


## Runtime và tests
Kiểm tra CUDA và chạy toàn bộ tests của source đã thêm hỗ trợ Hugging Face.


In [ ]:
run([PYTHON, '-c', "\nimport json, os, platform, sys\nfrom pathlib import Path\nimport torch, torchvision\nfrom importlib.metadata import distribution, version\nfrom evaluation.evidence import TRACE_SCHEMA_VERSION, SUMMARY_SCHEMA_VERSION\nassert sys.version_info[:2] == (3, 11)\nassert torch.__version__.split('+')[0] == '2.4.1'\nassert torchvision.__version__.split('+')[0] == '0.19.1'\nassert torch.version.cuda == '12.1' and torch.cuda.is_available(), 'CUDA 12.1 runtime unavailable'\nassert (TRACE_SCHEMA_VERSION, SUMMARY_SCHEMA_VERSION) == (3, 4)\nimport subprocess\nfrom runtime.experiment_matrix import build_experiment_matrix, build_command\nprobe_run = build_experiment_matrix(datasets=('CIFAR100C',), streams=('block',),\n                                   methods=('NoAdapt',), seeds=(0,), device='cuda')[0]\nchild_python = build_command(probe_run)[0]\nassert child_python == sys.executable, 'Generated command changed the virtualenv interpreter'\nsubprocess.run([child_python, '-c', 'import torch; assert torch.cuda.is_available(); print(torch.__version__)'], check=True)\nclip_source = json.loads(distribution('clip').read_text('direct_url.json'))\nassert clip_source['vcs_info']['commit_id'] == 'd05afc436d78f1c48dc0dbf8e5980a9d471f35f6'\nfor package, expected in [('numpy','1.26.4'),('pillow','10.4.0'),('pyyaml','6.0.2'),('tqdm','4.66.5')]:\n    assert version(package) == expected, package\nidentity = dict(python=sys.version, torch=torch.__version__, torchvision=torchvision.__version__,\n                cuda=torch.version.cuda, gpu=torch.cuda.get_device_name(0), platform=platform.platform(),\n                total_vram_gib=torch.cuda.get_device_properties(0).total_memory / 2**30,\n                visible_devices=os.environ.get('CUDA_VISIBLE_DEVICES'), clip_source=clip_source)\n(Path(os.environ['RAMEN_RUNTIME_ROOT'])/'device.json').write_text(json.dumps(identity,indent=2))\nprint(json.dumps(identity, indent=2))\n"], cwd=REPO, log=RUNTIME / 'runtime-check.log')

focused = ["tests.test_by_sample_normalization", "tests.test_ramen_cuda_half", "tests.test_entropy_gated_ramen", "tests.test_consensus_ramen",
           "tests.test_oracle_id_gradient_ramen", "tests.test_oracle_consensus_ramen",
           "tests.test_open_set", "tests.test_open_set_metrics", "tests.test_open_set_consensus_analysis",
           "tests.test_ordered_stream_evidence", "tests.test_experiment_matrix"]
run([PYTHON, "-m", "unittest", *focused], cwd=REPO, log=RUNTIME / "focused-tests.log")
run([PYTHON, "-m", "unittest", "discover", "-s", "tests", "-p", "test_*.py"],
    cwd=REPO, log=RUNTIME / "full-tests.log")
(RUNTIME / "test-status.json").write_text(json.dumps({"focused_exit": 0, "full_exit": 0}))


## Dữ liệu từ Hugging Face
Tải 95 Parquet ở revision cố định. Mỗi file NumPy sau chuyển đổi phải khớp checksum gốc; mọi lỗi dừng notebook.


In [ ]:
support_script = RUNTIME / "download-support.py"
support_script.write_text('"""Prepare the checksum-verified official CIFAR-100-C archive and CLIP model."""\nimport json\nimport os\nfrom pathlib import Path, PurePosixPath\nimport subprocess\nimport tarfile\nimport tempfile\nimport time\n\nfrom runtime.artifact_provenance import (\n    CIFAR100C_OFFICIAL_ACQUISITION, verify_official_cifar100c_archive,\n    generate_cifar100c_provenance, verify_cifar100c_provenance,\n    resolve_clip_model, verify_clip_checkpoint, ProvenanceError,\n)\nfrom evaluation.evidence import atomic_write_json\n\n\ndef preserve(path):\n    saved = path.with_name(f"{path.name}.rejected-{time.time_ns()}")\n    path.rename(saved)\n    print(f"Preserved unusable download: {saved}", flush=True)\n\n\ndef download(urls, path, verify, *, log_path=None, attempts=6, retry_delay=30):\n    """Resume transfers, rotate endpoints and publish only checksum-verified bytes."""\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    if path.exists():\n        try:\n            return verify(path)\n        except ProvenanceError:\n            preserve(path)\n    partial = path.with_suffix(path.suffix + ".part")\n    for attempt in range(attempts):\n        url = urls[attempt % len(urls)]\n        print(f"Download {path.name}: attempt {attempt + 1}/{attempts}: {url}", flush=True)\n        result = subprocess.run([\n            "curl", "--fail", "--location", "--connect-timeout", "30",\n            "--max-time", "3600", "--speed-time", "90", "--speed-limit", "1024",\n            "--continue-at", "-", "--output", str(partial),\n            "--write-out", "%{http_code}", url,\n        ], stdout=subprocess.PIPE, text=True)\n        http_status = result.stdout.strip()\n        record = {"file": path.name, "url": url, "attempt": attempt + 1,\n                  "curl_exit": result.returncode, "http_status": http_status,\n                  "verified": False}\n        verified = False\n        if partial.is_file() and (result.returncode == 0 or http_status == "416"):\n            try:\n                verify(partial)\n            except ProvenanceError as exc:\n                record["verification_error"] = str(exc)\n                preserve(partial)\n            else:\n                partial.replace(path)\n                verified = record["verified"] = True\n        elif result.returncode == 33 and partial.exists():\n            # This endpoint cannot resume; make the next attempt a fresh transfer.\n            preserve(partial)\n        if log_path is not None:\n            with Path(log_path).open("a") as log:\n                log.write(json.dumps(record) + "\\n")\n        if verified:\n            return verify(path)\n        if attempt + 1 < attempts:\n            print(f"Download incomplete (HTTP {http_status}); retry in {retry_delay}s.", flush=True)\n            time.sleep(retry_delay)\n    raise RuntimeError(\n        f"Could not obtain verified {path.name} after {attempts} attempts. "\n        "Keep the partial file and retry this cell, or attach the official archive via ARCHIVE_INPUT."\n    )\n\n\ndef prepare():\n    data = Path(os.environ["RAMEN_DATA_ROOT"])\n    runtime = Path(os.environ["RAMEN_EVIDENCE_ROOT"]) / "runtime"\n    runtime.mkdir(parents=True, exist_ok=True)\n    archive_input = os.environ.get("RAMEN_ARCHIVE_INPUT", "").strip()\n    archive = Path(archive_input) if archive_input else data.parent / "CIFAR-100-C.tar"\n    if archive_input and not archive.is_file():\n        raise FileNotFoundError(f"Attached archive not found: {archive}")\n    if not archive_input:\n        download([\n            "https://zenodo.org/records/3555552/files/CIFAR-100-C.tar?download=1",\n            CIFAR100C_OFFICIAL_ACQUISITION["url"],\n        ], archive, verify_official_cifar100c_archive,\n            log_path=runtime / "download-attempts.jsonl")\n    print("Verifying the official archive MD5 and size...", flush=True)\n    acquisition = verify_official_cifar100c_archive(archive)\n    atomic_write_json(runtime / "archive-acquisition.json", acquisition)\n    dataset = data / "corruption/CIFAR-100-C"\n    if not dataset.exists():\n        data.mkdir(parents=True, exist_ok=True)\n        with tempfile.TemporaryDirectory(prefix="extract-staging-", dir=data) as tmp:\n            staging = Path(tmp)\n            # Permit only the expected dataset tree and regular files/directories.\n            with tarfile.open(archive) as source:\n                members = source.getmembers()\n                for member in members:\n                    parts = PurePosixPath(member.name).parts\n                    if (not parts or parts[0] != "CIFAR-100-C" or ".." in parts\n                            or not (member.isfile() or member.isdir())):\n                        raise RuntimeError(f"Unexpected archive entry: {member.name}")\n                source.extractall(staging, members=members, filter="data")\n            staged = staging / "CIFAR-100-C"\n            # Inventory is created only for bytes extracted from the verified archive.\n            generate_cifar100c_provenance(staged, acquisition=acquisition)\n            dataset.parent.mkdir(parents=True, exist_ok=True)\n            staged.rename(dataset)\n    # On resume, never bless an arbitrary existing tree by rebuilding its sidecar.\n    dataset_provenance = verify_cifar100c_provenance(dataset, exact=True)\n    resolved = resolve_clip_model("clip_vitbase16")\n    cache = Path.home() / ".cache/clip"\n    model_provenance = download(\n        [resolved["url"]], cache / resolved["filename"],\n        lambda path: verify_clip_checkpoint("clip_vitbase16", path),\n        log_path=runtime / "download-attempts.jsonl",\n    )\n    atomic_write_json(runtime / "artifact-provenance.json", {\n        "dataset": dataset_provenance, "model": model_provenance,\n    })\n    print("Official data and model verified.", flush=True)\n\n\nif __name__ == "__main__":\n    prepare()\n')
prepare_script = RUNTIME / "prepare-huggingface-data.py"
prepare_script.write_text('"""Download pinned HF Parquet files and reconstruct checksum-identical CIFAR100C arrays."""\n\nfrom concurrent.futures import ThreadPoolExecutor\nimport importlib.util\nfrom io import BytesIO\nimport os\nfrom pathlib import Path\nimport time\n\nimport numpy as np\nimport pyarrow.parquet as pq\nfrom PIL import Image\nfrom huggingface_hub import hf_hub_download\n\nfrom evaluation.evidence import atomic_write_json\nfrom runtime.artifact_provenance import (\n    ProvenanceError, checksum_regular_file, generate_cifar100c_provenance,\n    verify_cifar100c_provenance, resolve_clip_model, verify_clip_checkpoint,\n)\nfrom runtime.cifar100c_huggingface import (\n    HF_REPOSITORY, HF_REVISION, CIFAR100C_NPY_MD5, CIFAR100C_HF_ACQUISITION,\n)\n\n\ndef verified_npy(path):\n    if not path.is_file():\n        return False\n    digest = checksum_regular_file(path, "md5")["checksum"]\n    if digest != CIFAR100C_NPY_MD5[path.name]:\n        raise ProvenanceError(f"Original CIFAR-100-C checksum mismatch: {path.name}")\n    return True\n\n\ndef fetch(corruption, severity, cache):\n    filename = f"data/{corruption}/severity_{severity}/data-00000.parquet"\n    for attempt in range(1, 7):\n        try:\n            return Path(hf_hub_download(\n                HF_REPOSITORY, filename, repo_type="dataset", revision=HF_REVISION,\n                cache_dir=str(cache), token=False,\n            ))\n        except Exception as exc:\n            if attempt == 6:\n                raise\n            print(f"Retry {filename} ({attempt}/6): {type(exc).__name__}: {exc}", flush=True)\n            time.sleep(30)\n\n\ndef write_severity(parquet, output, offset, expected_labels=None):\n    """Preserve the original row order; refuse wrong counts, labels, shapes or modes."""\n    source = pq.ParquetFile(parquet)\n    if source.metadata.num_rows != 10000 or set(source.schema_arrow.names) != {"image", "label"}:\n        raise ValueError(f"Unexpected CIFAR100C Parquet schema/row count: {parquet}")\n    labels, row = np.empty(10000, dtype=np.uint8), 0\n    for batch in source.iter_batches(batch_size=256):\n        values = batch.to_pydict()\n        for encoded, label in zip(values["image"], values["label"]):\n            if not isinstance(label, int) or isinstance(label, bool) or not 0 <= label < 100:\n                raise ValueError(f"Invalid CIFAR100C label at row {row}: {label}")\n            if not isinstance(encoded, dict) or not encoded.get("bytes"):\n                raise ValueError(f"Missing embedded image bytes at row {row}")\n            with Image.open(BytesIO(encoded["bytes"])) as image:\n                if image.mode != "RGB" or image.size != (32, 32):\n                    raise ValueError(f"Unexpected CIFAR100C image at row {row}")\n                output[offset + row] = np.asarray(image, dtype=np.uint8)\n            labels[row] = label\n            row += 1\n    if row != 10000:\n        raise ValueError("Incomplete severity split")\n    if expected_labels is not None and not np.array_equal(labels, expected_labels):\n        raise ValueError("CIFAR100C label order differs across severity/corruption files")\n    return labels\n\n\ndef convert_corruption(corruption, destination, cache, reference_labels=None):\n    output = destination / f"{corruption}.npy"\n    if verified_npy(output):\n        return reference_labels\n    # The .part file is never treated as a completed array on the next invocation.\n    partial = output.with_suffix(".npy.part")\n    array = np.lib.format.open_memmap(\n        partial, mode="w+", dtype=np.uint8, shape=(50000, 32, 32, 3), version=(1, 0),\n    )\n    try:\n        with ThreadPoolExecutor(max_workers=3) as pool:\n            paths = list(pool.map(lambda level: fetch(corruption, level, cache), range(1, 6)))\n        for severity, parquet in enumerate(paths, 1):\n            print(f"  convert {corruption}, severity {severity}/5", flush=True)\n            labels = write_severity(parquet, array, (severity - 1) * 10000, reference_labels)\n            if reference_labels is None:\n                reference_labels = labels\n        array.flush()\n    finally:\n        del array\n    actual = checksum_regular_file(partial, "md5")["checksum"]\n    if actual != CIFAR100C_NPY_MD5[output.name]:\n        raise ProvenanceError(f"Reconstructed array differs from the original: {output.name}")\n    partial.replace(output)\n    return reference_labels\n\n\ndef prepare():\n    data = Path(os.environ["RAMEN_DATA_ROOT"])\n    runtime = Path(os.environ["RAMEN_RUNTIME_ROOT"])\n    data.mkdir(parents=True, exist_ok=True)\n    runtime.mkdir(parents=True, exist_ok=True)\n    dataset = data / "corruption/CIFAR-100-C"\n    cache = data.parent / "nb-ramen-hf-cache"\n    if dataset.exists():\n        report = verify_cifar100c_provenance(dataset, exact=True)\n        if report["acquisition"] != CIFAR100C_HF_ACQUISITION:\n            raise RuntimeError("This notebook requires its separately verified Hugging Face dataset")\n    else:\n        staging = data / "cifar100c-hf-staging"\n        staging.mkdir(exist_ok=True)\n        labels_path = staging / "labels.npy"\n        labels = None\n        if verified_npy(labels_path):\n            labels = np.load(labels_path, allow_pickle=False)[:10000]\n        corruptions = sorted(Path(name).stem for name in CIFAR100C_NPY_MD5 if name != "labels.npy")\n        for index, corruption in enumerate(corruptions, 1):\n            print(f"[{index}/19] {corruption}", flush=True)\n            # If conversion completed before a crash but labels were not saved,\n            # recover labels from a pinned split instead of accepting missing labels.\n            if labels is None and (staging / f"{corruption}.npy").exists():\n                raw = pq.read_table(fetch(corruption, 1, cache), columns=["label"])["label"].to_numpy()\n                if raw.shape != (10000,) or raw.dtype.kind not in "iu" or np.any((raw < 0) | (raw >= 100)):\n                    raise ValueError("Invalid or incomplete label split")\n                labels = raw.astype(np.uint8)\n            labels = convert_corruption(corruption, staging, cache, labels)\n            if not labels_path.exists():\n                partial = labels_path.with_suffix(".npy.part")\n                with partial.open("wb") as handle:\n                    np.save(handle, np.tile(labels, 5), allow_pickle=False)\n                if checksum_regular_file(partial, "md5")["checksum"] != CIFAR100C_NPY_MD5["labels.npy"]:\n                    raise ProvenanceError("Original CIFAR100C labels/order checksum mismatch")\n                partial.replace(labels_path)\n        # The patched generator independently rechecks all original file MD5s.\n        generate_cifar100c_provenance(staging, acquisition=CIFAR100C_HF_ACQUISITION)\n        dataset.parent.mkdir(parents=True, exist_ok=True)\n        staging.rename(dataset)\n        report = verify_cifar100c_provenance(dataset, exact=True)\n\n    atomic_write_json(runtime / "huggingface-acquisition.json", report["acquisition"])\n    support_path = runtime / "download-support.py"\n    spec = importlib.util.spec_from_file_location("download_support", support_path)\n    support = importlib.util.module_from_spec(spec)\n    spec.loader.exec_module(support)\n    resolved = resolve_clip_model("clip_vitbase16")\n    model = support.download(\n        [resolved["url"]], Path.home() / ".cache/clip" / resolved["filename"],\n        lambda path: verify_clip_checkpoint("clip_vitbase16", path),\n        log_path=runtime / "download-attempts.jsonl",\n    )\n    atomic_write_json(runtime / "artifact-provenance.json", {"dataset": report, "model": model})\n    print("Hugging Face data matches all original NPY checksums; CLIP verified.", flush=True)\n\n\nif __name__ == "__main__":\n    prepare()\n')
run([PYTHON, prepare_script], cwd=REPO, log=RUNTIME / "prepare-data.log")
run([PYTHON, "-m", "runtime.preflight", "--data-root", DATA,
     "--dataset", "CIFAR100C", "--deep", "--json"],
    cwd=REPO, log=RUNTIME / "cifar100c-deep-preflight.json")


## Thực thi full matrix và resume

Run NoAdapt trước các phương pháp cùng cell. Mỗi run hoàn tất phải qua strict
validation và khớp revision/config/fingerprint baseline, rồi mới ghi checkpoint.
Các run đủ summary nhưng không hợp lệ gây lỗi, không bị bỏ qua.
Run bị ngắt và chưa có summary được chuyển vào `interrupted/` để giữ chẩn đoán,
sau đó chạy lại nguyên run. Không nối tiếp từ giữa một stream.

Logs từng run ở `runtime/<session>/`; status toàn matrix ở `status.json`.
Khi đạt **252/252**, tạo `analysis/open-set-consensus.json`, `per-cell-metrics.csv`
và README. Chỉ `status="complete"` cùng `full_matrix_complete=true` mới là full evidence.
Trạng thái `paused_session_budget` cần tiếp tục ở phiên sau.


In [ ]:
full_script = RUNTIME / 'run-full-matrix.py'
full_script.write_text('"""Run the frozen canonical 252-run matrix with per-run checkpointing."""\n\nimport argparse\nimport csv\nimport importlib.util\nimport json\nimport os\nfrom pathlib import Path\nimport subprocess\nimport sys\nimport time\n\nfrom evaluation.evidence import atomic_write_json, _source_tree_fingerprint\nfrom evaluation.open_set_consensus_analysis import analyse_open_set_completed_runs\nfrom runtime.experiment_matrix import build_canonical_open_set_evidence_matrix, build_command, validate_completed_run\n\n\nREVISION = "dc3cbf02215cff59046f048e6580dd8f8d40af89"\n\n\ndef load_checkpoint_helper():\n    path = Path(__file__).with_name("full-run-checkpoint.py")\n    spec = importlib.util.spec_from_file_location("full_run_checkpoint", path)\n    module = importlib.util.module_from_spec(spec)\n    spec.loader.exec_module(module)\n    return module.checkpoint\n\n\ndef verify_run(run, source_identity, baselines):\n    evidence = validate_completed_run(run)\n    git = evidence["manifest"]["git"]\n    if git.get("commit") != REVISION or git.get("dirty") is not False or git.get("source") != source_identity:\n        raise RuntimeError(f"Run source identity differs from the frozen revision: {run.run_id}")\n    fingerprint = evidence["summary"]["stream_fingerprint"]\n    if run.reference_trace is not None:\n        if baselines.get(run.reference_trace) != fingerprint:\n            raise RuntimeError(f"Paired NoAdapt is absent or has a different stream: {run.run_id}")\n    else:\n        baselines[run.run_dir / "trace.jsonl"] = fingerprint\n    # Keep summaries/manifests for descriptive analysis, not all stream exports in RAM.\n    return {"manifest": evidence["manifest"], "summary": evidence["summary"]}\n\n\ndef run_command(command, repo, log):\n    with log.open("w") as handle:\n        process = subprocess.Popen(command, cwd=repo, stdout=handle, stderr=subprocess.STDOUT)\n        started = time.monotonic()\n        try:\n            while True:\n                try:\n                    code = process.wait(timeout=60)\n                    break\n                except subprocess.TimeoutExpired:\n                    print(f"Still running ({time.monotonic() - started:.0f}s); log: {log.name}", flush=True)\n        except BaseException:\n            process.terminate()\n            try:\n                process.wait(timeout=15)\n            except subprocess.TimeoutExpired:\n                process.kill()\n                process.wait()\n            raise\n    if code:\n        with log.open() as handle:\n            from collections import deque\n            print("".join(deque(handle, maxlen=35)), flush=True)\n        raise subprocess.CalledProcessError(code, command)\n\n\ndef write_analysis(root, completed):\n    report = analyse_open_set_completed_runs(completed)\n    if report["classification"] != "canonical_cuda_expected" or not report["coverage"]["complete"]:\n        raise RuntimeError("All 252 runs must satisfy the canonical analysis contract")\n    destination = root / "analysis"\n    destination.mkdir(exist_ok=True)\n    atomic_write_json(destination / "open-set-consensus.json", report)\n    fields = ["ood_ratio", "stream_mode", "seed", "method", "id_accuracy", "auroc", "fpr95", "h_score"]\n    with (destination / "per-cell-metrics.csv").open("w", newline="") as handle:\n        writer = csv.DictWriter(handle, fieldnames=fields)\n        writer.writeheader()\n        for cell in report["comparisons"]:\n            for method, metrics in cell["methods"].items():\n                writer.writerow({**{key: cell[key] for key in fields[:3]}, "method": method,\n                                 **{key: metrics.get(key) for key in fields[4:]}})\n    (destination / "README.md").write_text(\n        "# Complete canonical CIFAR-100-C evidence\\n\\n"\n        f"Revision: `{REVISION}`. All 252 full-stream runs passed strict validation.\\n\\n"\n        "Read open-set-consensus.json for paired metrics, oracle diagnostics, stability and costs; "\n        "per-cell-metrics.csv retains all three seeds and each stream/OOD ratio separately. "\n        "Undefined OOD detection metrics at OOD=0 remain empty/null.\\n\\n"\n        "This is descriptive evidence, not a certification that Consensus improves performance. "\n        "It covers the primary CIFAR-100-C matrix; DomainNet and split-robustness studies are separate.\\n"\n    )\n\n\ndef main():\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("--plan-only", action="store_true")\n    parser.add_argument("--max-runs", type=int, default=0, help="New complete runs this invocation; 0 means all remaining.")\n    parser.add_argument("--session-hours", type=float, default=8.0, help="Stop between runs after this budget; 0 disables it.")\n    args = parser.parse_args()\n    if args.max_runs < 0 or not 0 <= args.session_hours < float("inf"):\n        parser.error("budgets must be finite and nonnegative")\n    repo = Path(os.environ["RAMEN_REPOSITORY"]).resolve()\n    data = Path(os.environ["RAMEN_DATA_ROOT"]).resolve()\n    root = Path(os.environ["RAMEN_EVIDENCE_ROOT"]).resolve()\n    runtime = Path(os.environ["RAMEN_RUNTIME_ROOT"]).resolve()\n    runtime.mkdir(parents=True, exist_ok=True)\n    runs = build_canonical_open_set_evidence_matrix(\n        data_root=data, evidence_dir=root / "canonical", config_dir=repo / "cfg",\n        device="cuda", artifact_provenance="fast",\n    )\n    atomic_write_json(runtime / "canonical-plan.json", {\n        "execute": not args.plan_only, "runs": [run.to_dict() for run in runs],\n        "commands": [build_command(run, python_executable=sys.executable) for run in runs],\n    })\n    print(f"Canonical plan: {len(runs)} full-stream runs, B=100, 400 source samples/domain.", flush=True)\n    if args.plan_only:\n        return\n    import torch\n    if not torch.cuda.is_available():\n        raise RuntimeError("A real CUDA device is required")\n    head = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=repo, text=True).strip()\n    dirty = subprocess.check_output(["git", "status", "--porcelain"], cwd=repo, text=True)\n    if head != REVISION or os.environ.get("RAMEN_REVISION") != REVISION or dirty:\n        raise RuntimeError("Use the clean frozen experiment revision")\n    tests = json.loads((runtime / "test-status.json").read_text())\n    preflight = json.loads((runtime / "cifar100c-deep-preflight.json").read_text())\n    if tests != {"focused_exit": 0, "full_exit": 0} or not preflight.get("valid"):\n        raise RuntimeError("Pass this session\'s tests and deep preflight first")\n    source_identity = _source_tree_fingerprint(repo)\n    if source_identity is None:\n        raise RuntimeError("Cannot fingerprint experiment source")\n    device = json.loads((runtime / "device.json").read_text())\n    identity = {"revision": REVISION, "data_root": str(data), "runs": 252,\n                "runtime": {key: device[key] for key in ("gpu", "torch", "torchvision", "cuda")}}\n    campaign_path = root / "campaign.json"\n    if campaign_path.exists() and json.loads(campaign_path.read_text()) != identity:\n        raise RuntimeError("Campaign identity changed; use the same revision, data path, GPU type and runtime")\n    atomic_write_json(campaign_path, identity)\n    checkpoint = load_checkpoint_helper()\n    completed, baselines = [], {}\n    state = {"status": "running", "revision": REVISION, "planned": 252, "completed": [],\n             "new_runs_this_session": 0, "session": runtime.name, "full_matrix_complete": False}\n    started = time.monotonic()\n    failed = False\n    try:\n        for index, run in enumerate(runs, 1):\n            new_run = False\n            state["current_run"] = run.run_id\n            if run.run_dir.exists() and not (run.run_dir / "summary.json").exists():\n                # Preserve interrupted work; restart this whole run, never append a partial trace.\n                interrupted = root / "interrupted" / runtime.name\n                interrupted.mkdir(parents=True, exist_ok=True)\n                preserved = interrupted / f"{run.run_id}-{time.time_ns()}"\n                run.run_dir.rename(preserved)\n                print(f"Preserved incomplete run under {preserved}", flush=True)\n            if run.run_dir.exists():\n                checked = verify_run(run, source_identity, baselines)\n                print(f"[{index}/252] validated resume: {run.method} {run.stream_mode} seed={run.seed} OOD={run.ood_ratio}", flush=True)\n            else:\n                if ((args.max_runs and state["new_runs_this_session"] >= args.max_runs)\n                        or (args.session_hours and time.monotonic() - started >= args.session_hours * 3600)):\n                    state["status"] = "paused_session_budget"\n                    break\n                if run.reference_trace is not None and run.reference_trace not in baselines:\n                    raise RuntimeError("No validated paired NoAdapt before adapted run")\n                print(f"[{index}/252] RUN {run.method} {run.stream_mode} seed={run.seed} OOD={run.ood_ratio}", flush=True)\n                atomic_write_json(root / "status.json", state)\n                run_command(build_command(run, python_executable=sys.executable), repo, runtime / f"{run.run_id}.log")\n                checked = verify_run(run, source_identity, baselines)\n                state["new_runs_this_session"] += 1\n                new_run = True\n            completed.append((run, checked))\n            state["completed"].append(run.run_id)\n            state["current_run"] = None\n            atomic_write_json(root / "status.json", state)\n            if new_run:\n                # Per-run atomic snapshots retain the last complete ZIP if the session is killed.\n                print(f"Checkpoint: {checkpoint(root)}", flush=True)\n        if len(completed) == 252:\n            write_analysis(root, completed)\n            state.update(status="complete", full_matrix_complete=True)\n    except BaseException as exc:\n        failed = True\n        state.update(status="failed", error=f"{type(exc).__name__}: {exc}")\n        raise\n    finally:\n        atomic_write_json(root / "status.json", state)\n        try:\n            print(f"Evidence archive: {checkpoint(root)}", flush=True)\n        except Exception as exc:\n            if not failed:\n                raise\n            print(f"Checkpoint failed too: {exc}. The previous complete ZIP is retained.", flush=True)\n    print(json.dumps({key: value for key, value in state.items() if key != "completed"}, indent=2), flush=True)\n    print(f"Validated {len(completed)}/252. Download the ZIP before ending the session.", flush=True)\n\n\nif __name__ == "__main__":\n    main()\n')
checkpoint_script = RUNTIME / 'full-run-checkpoint.py'
checkpoint_script.write_text('"""Atomic ZIP checkpoints and non-overwriting restore for Kaggle campaigns."""\n\nimport argparse\nimport hashlib\nimport json\nfrom pathlib import Path, PurePosixPath\nimport shutil\nimport stat\nimport tempfile\nimport zipfile\n\n\ndef checkpoint(evidence):\n    evidence = Path(evidence).resolve()\n    if not evidence.is_dir():\n        raise FileNotFoundError(evidence)\n    archive = evidence.with_suffix(".zip")\n    partial = archive.with_suffix(".zip.part")\n    # Keep the previous complete ZIP until the replacement is closed successfully.\n    try:\n        with zipfile.ZipFile(partial, "w", zipfile.ZIP_DEFLATED, compresslevel=1) as output:\n            for path in sorted(evidence.rglob("*")):\n                if path.is_symlink():\n                    raise ValueError(f"Evidence must not contain symlinks: {path}")\n                if path.is_file():\n                    output.write(path, path.relative_to(evidence.parent).as_posix())\n        partial.replace(archive)\n    except BaseException:\n        partial.unlink(missing_ok=True)\n        raise\n    return archive\n\n\ndef restore(archive, evidence):\n    archive, evidence = Path(archive), Path(evidence)\n    with archive.open("rb") as handle:\n        digest = hashlib.file_digest(handle, "sha256").hexdigest()\n    marker_name = "restored-archive.json"\n    if evidence.exists():\n        marker = evidence / marker_name\n        if marker.is_file() and json.loads(marker.read_text()).get("sha256") == digest:\n            return  # Setup rerun: retain all progress since the same restore.\n        raise FileExistsError(\n            "Evidence already exists. Clear RESUME_ARCHIVE to use it; restore never merges or overwrites runs."\n        )\n    evidence.parent.mkdir(parents=True, exist_ok=True)\n    with zipfile.ZipFile(archive) as source:\n        seen = set()\n        for entry in source.infolist():\n            parts = PurePosixPath(entry.filename).parts\n            mode = entry.external_attr >> 16\n            normalized = PurePosixPath(entry.filename).as_posix()\n            if (not parts or parts[0] != evidence.name or ".." in parts\n                    or "\\\\" in entry.filename or PurePosixPath(entry.filename).is_absolute()\n                    or stat.S_ISLNK(mode) or normalized in seen):\n                raise ValueError(f"Invalid checkpoint member: {entry.filename}")\n            seen.add(normalized)\n        if not seen:\n            raise ValueError("Checkpoint is empty")\n        required = sum(entry.file_size for entry in source.infolist())\n        if required >= shutil.disk_usage(evidence.parent).free:\n            raise OSError("Insufficient disk space to restore this checkpoint")\n        with tempfile.TemporaryDirectory(prefix="restore-full-", dir=evidence.parent) as tmp:\n            # All paths/types were checked before any extraction. CRC errors abort staging.\n            source.extractall(tmp)\n            staged = Path(tmp) / evidence.name\n            if not staged.is_dir():\n                raise ValueError("Checkpoint root must be a directory")\n            (staged / marker_name).write_text(json.dumps({"sha256": digest}, indent=2) + "\\n")\n            staged.rename(evidence)\n\n\nif __name__ == "__main__":\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("action", choices=("save", "restore"))\n    parser.add_argument("evidence", type=Path)\n    parser.add_argument("--archive", type=Path)\n    args = parser.parse_args()\n    if args.action == "restore":\n        if args.archive is None:\n            parser.error("restore requires --archive")\n        restore(args.archive, args.evidence)\n        print("Checkpoint restored; each run still requires strict validation.")\n    else:\n        print(checkpoint(args.evidence))\n')

try:
    run([PYTHON, full_script, "--max-runs", str(MAX_RUNS), "--session-hours", str(SESSION_HOURS)],
        cwd=REPO, log=RUNTIME / "full-execution.log")
finally:
    run([PYTHON, checkpoint_script, "save", EVIDENCE])


## Tải evidence / tiếp tục phiên sau

Tải `nb-ramen-full-hf-dc3cbf0.zip`. Để resume phiên mới, upload ZIP đã tải thành Kaggle
Input và điền `RESUME_ARCHIVE` ở cell đầu. Nếu đang tiếp tục ngay trong phiên
hiện tại, để trống `RESUME_ARCHIVE`: notebook dùng evidence đang có.
File Input cần là ZIP full đúng tên root, không phải ZIP smoke hay chỉ vài summary.
Nếu Input đã được giải nén tự động, nén lại cây `nb-ramen-full-hf-dc3cbf0/` thành ZIP
với đúng một root đó trước khi dùng. Không merge các checkpoint bằng tay.

Ngân sách/hết phiên có thể buộc chạy lại run đang dở, nhưng các run đã checkpoint
và được validator chấp nhận sẽ được giữ. Sau 252/252, gửi ZIP để đánh giá kết quả.


In [ ]:
status_path = EVIDENCE / "status.json"
if status_path.exists():
    status = json.loads(status_path.read_text())
    print("Status:", status["status"], "| Validated:", len(status["completed"]), "/", status["planned"])
    if status.get("error"):
        print(status["error"])
archive = EVIDENCE.with_suffix(".zip")
if not archive.exists():
    run([sys.executable, checkpoint_script, "save", EVIDENCE])
from IPython.display import display, FileLink
display(FileLink(str(archive)))
